[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/notebooks/show3dvolume/show3dvolume_dual.ipynb)
# Show3DVolume -- Dual-Volume Comparison
Side-by-side comparison of two 3D volumes with shared slice navigation.
Useful for comparing reconstruction vs ground truth, before vs after processing,
or two different reconstruction algorithms on the same specimen.

In [ ]:
# Install in Google Colab
try:
    import google.colab
    !pip install -q -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ quantem-widget
except ImportError:
    pass  # Not in Colab, skip

In [ ]:
try:
    %load_ext autoreload
    %autoreload 2
    %env ANYWIDGET_HMR=1
except Exception:
    pass  # autoreload unavailable (Colab Python 3.12+)

In [ ]:
import numpy as np
import torch
import quantem.widget
from quantem.widget import Show3DVolume, profile
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"quantem.widget {quantem.widget.__version__}, device={device}")
profile()

## Generate synthetic data
A gold nanoparticle "ground truth" and a noisy "reconstruction" with missing wedge artifact,
simulating a real electron tomography comparison workflow.

In [ ]:
def make_dual_volumes(n=80):
    """Ground truth nanoparticle + noisy reconstruction with missing wedge."""
    coords = torch.arange(n, dtype=torch.float32, device=device)
    z, y, x = torch.meshgrid(coords, coords, coords, indexing="ij")
    cz, cy, cx = n / 2, n / 2, n / 2
    r = torch.sqrt((x - cx) ** 2 + (y - cy) ** 2 + (z - cz) ** 2)
    R = n * 0.35
    # Truncated octahedron shape
    ax, ay, az = torch.abs(x - cx), torch.abs(y - cy), torch.abs(z - cz)
    l1 = ax + ay + az
    linf = torch.maximum(torch.maximum(ax, ay), az)
    shape_dist = torch.maximum(l1 / 1.6, linf)
    particle = 1.0 / (1 + torch.exp((shape_dist - R) * 1.2))
    # Core-shell structure
    core = 1.0 / (1 + torch.exp((r - R * 0.5) * 0.8))
    density = 0.6 + 0.4 * core
    # Lattice fringes
    d111 = 3.5
    fringes = 0.1 * torch.cos(2 * np.pi * (x + y + z) / (d111 * np.sqrt(3)))
    gt = particle * (density + fringes)
    gt_np = gt.cpu().numpy()
    gt_np = np.clip(gt_np, 0, None).astype(np.float32)
    # Reconstruction: add noise + missing wedge elongation
    recon = gt_np.copy()
    # Missing wedge: blur along Z (simulates limited tilt range)
    from scipy.ndimage import gaussian_filter1d
    recon = gaussian_filter1d(recon, sigma=1.5, axis=0)
    # Add reconstruction noise
    recon += np.random.normal(0, 0.03, recon.shape).astype(np.float32)
    recon = np.clip(recon, 0, None).astype(np.float32)
    return gt_np, recon
ground_truth, reconstruction = make_dual_volumes()
print(f"Ground truth: {ground_truth.shape}, range [{ground_truth.min():.4f}, {ground_truth.max():.4f}]")
print(f"Reconstruction: {reconstruction.shape}, range [{reconstruction.min():.4f}, {reconstruction.max():.4f}]")

## 1. Dual-volume comparison
Both 3D renderers and all three orthogonal slice panels are shown side by side.
Sliders, zoom/pan, and playback are shared — stats and cursor readout are independent.

In [ ]:
w = Show3DVolume(
    ground_truth,
    data_b=reconstruction,
    title="Ground Truth",
    title_b="Reconstruction",
    cmap="inferno",
)
w

In [ ]:
w.summary()

In [ ]:
repr(w)

## 2. FFT comparison
Enable FFT to compare the frequency content of both volumes.
Missing wedge artifacts show as elongated streaks in the reconstruction's FFT.

In [ ]:
w_fft = Show3DVolume(
    ground_truth,
    data_b=reconstruction,
    title="GT",
    title_b="Recon",
    cmap="inferno",
    show_fft=True,
)
w_fft

## 3. Auto contrast
When the two volumes have different intensity ranges, auto contrast independently
stretches each for easier visual comparison.

In [ ]:
Show3DVolume(
    ground_truth,
    data_b=reconstruction,
    title="GT (auto)",
    title_b="Recon (auto)",
    cmap="viridis",
    auto_contrast=True,
)

## 4. Replace data with `set_image()`
Swap both volumes without recreating the widget — all display settings are preserved.

In [ ]:
gt2, recon2 = make_dual_volumes(64)
w.set_image(gt2, data_b=recon2)
print(f"Updated to {w.nz}x{w.ny}x{w.nx}, dual={w.dual_mode}")

## 5. State persistence
Save and restore display settings including dual-mode titles.

In [ ]:
import json
from pathlib import Path
w_save = Show3DVolume(
    ground_truth, data_b=reconstruction,
    title="GT", title_b="Recon",
    cmap="viridis", log_scale=True,
)
w_save.slice_z = 40
sd = w_save.state_dict()
print(f"dual_mode={sd['dual_mode']}, title_b={sd['title_b']}")
w_save.save("dual_state.json")
# Restore
w_loaded = Show3DVolume(ground_truth, data_b=reconstruction, state="dual_state.json")
print(f"Restored: cmap={w_loaded.cmap}, slice_z={w_loaded.slice_z}")
Path("dual_state.json").unlink(missing_ok=True)